# 🌦️ Project 3: Global Weather Data Analysis
**Multi-Domain Data Analysis Portfolio — Weather & Climate Domain**

**Author:** Binal Doshi | MSc AI & Data Science, University of Mumbai  
**Dataset:** Global Weather Repository (2018–2023) — 10 Cities, 21,910 Records  
**Tools:** Python · Pandas · Matplotlib · Seaborn · SciPy  

---

## Project Objectives
1. Analyze temperature trends across major global cities (2018–2023)
2. Study rainfall patterns — seasonal distribution & year-on-year trends
3. Identify extreme weather events and their geographic patterns
4. Examine relationships between weather variables (humidity, wind, pressure)
5. Extract actionable insights for climate understanding and urban planning

---


In [ ]:
# ─── CELL 1: Imports & Setup ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['font.family'] = 'DejaVu Sans'

print("✅ Libraries loaded successfully")
print(f"  pandas  {pd.__version__}")
print(f"  numpy   {np.__version__}")
print(f"  seaborn {sns.__version__}")


## 1. Data Loading & Exploration

In [ ]:
# ─── CELL 2: Load Dataset ────────────────────────────────────────────────────
df = pd.read_csv('../data/global_weather_2018_2023.csv', parse_dates=['date'])

# Feature engineering
df['year']       = df['date'].dt.year
df['month']      = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')
df['season']     = df['month'].map({
    12:'Winter', 1:'Winter',  2:'Winter',
     3:'Spring', 4:'Spring',  5:'Spring',
     6:'Summer', 7:'Summer',  8:'Summer',
     9:'Autumn',10:'Autumn', 11:'Autumn'
})

print(f"📦 Dataset Shape : {df.shape}")
print(f"📅 Date Range    : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"🌍 Cities        : {df['city'].nunique()} → {list(df['city'].unique())}")
print(f"🔢 Total Records : {len(df):,}")
print()
print("Column Info:")
print(df.dtypes)


In [ ]:
# ─── CELL 3: Data Quality Check ──────────────────────────────────────────────
print("=" * 50)
print("MISSING VALUES:")
print(df.isnull().sum())
print()
print("DUPLICATES:", df.duplicated().sum())
print()
print("DESCRIPTIVE STATISTICS:")
print(df[['temp_avg_c','temp_max_c','temp_min_c','precipitation_mm',
          'humidity_pct','wind_speed_kmh','pressure_hpa','uv_index']].describe().round(2))


## 2. Data Cleaning & Validation

In [ ]:
# ─── CELL 4: Data Cleaning ───────────────────────────────────────────────────
original_shape = df.shape

# Remove physical impossibilities
df = df[df['temp_max_c'] >= df['temp_min_c']]
df = df[df['precipitation_mm'] >= 0]
df = df[df['humidity_pct'].between(0, 100)]
df = df[df['wind_speed_kmh'] >= 0]

print(f"Before cleaning : {original_shape[0]:,} rows")
print(f"After cleaning  : {df.shape[0]:,} rows")
print(f"Rows removed    : {original_shape[0] - df.shape[0]}")
print()

# Summary statistics per city
city_summary = df.groupby('city').agg(
    Records     = ('date', 'count'),
    Avg_Temp    = ('temp_avg_c',      'mean'),
    Max_Temp    = ('temp_max_c',      'max'),
    Min_Temp    = ('temp_min_c',      'min'),
    Total_Rain  = ('precipitation_mm','sum'),
    Avg_Humidity= ('humidity_pct',    'mean'),
).round(2)
print("City Summary Statistics:")
print(city_summary.to_string())


## 3. Temperature Analysis

In [ ]:
# ─── CELL 5: VIZ 1 — Annual Temperature Trends ───────────────────────────────
palette = ['#1a6ea8','#e84545','#2ecc71','#f39c12','#9b59b6',
           '#1abc9c','#e67e22','#e74c3c','#3498db','#95a5a6']
city_pal = dict(zip(df['city'].unique(), palette))

yearly = df.groupby(['city','year'])['temp_avg_c'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14,6))
for city, grp in yearly.groupby('city'):
    ax.plot(grp['year'], grp['temp_avg_c'], marker='o', linewidth=2.2,
            label=city, color=city_pal[city])
ax.set_title('Annual Average Temperature Trends by City (2018–2023)',
             fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('Year'); ax.set_ylabel('Average Temperature (°C)')
ax.legend(ncol=5, fontsize=9, loc='upper center', bbox_to_anchor=(0.5,-0.18))
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('../visualizations/viz1_annual_temp_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Insight: Dubai and Singapore consistently show the highest temperatures.")
print("   Moscow and Delhi show the widest year-to-year variation.")


In [ ]:
# ─── CELL 6: VIZ 2 — Monthly Heatmap ─────────────────────────────────────────
pivot = df.groupby(['city','month'])['temp_avg_c'].mean().unstack()
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13,6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlBu_r',
            linewidths=0.5, ax=ax, cbar_kws={'label':'Avg Temp (°C)'})
ax.set_title('Monthly Average Temperature Heatmap by City',
             fontsize=15, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../visualizations/viz2_monthly_temp_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Insight: Moscow's January average is well below 0°C, while Singapore")
print("   maintains near-constant tropical warmth (~27°C) year-round.")


In [ ]:
# ─── CELL 7: VIZ 5 — Temperature Box Plot ────────────────────────────────────
order = df.groupby('city')['temp_avg_c'].mean().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(14,6))
sns.boxplot(data=df, x='city', y='temp_avg_c', order=order,
            palette={c: city_pal[c] for c in order}, ax=ax,
            linewidth=1.2, fliersize=2, flierprops={'alpha':0.3})
ax.set_title('Temperature Distribution by City (2018–2023)',
             fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('City (sorted by mean temp)'); ax.set_ylabel('Average Temperature (°C)')
plt.xticks(rotation=30, ha='right')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../visualizations/viz5_temp_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Insight: Moscow has the widest IQR (~30°C), reflecting extreme")
print("   continental climate. Dubai and Singapore have the tightest spread.")


## 4. Rainfall Analysis

In [ ]:
# ─── CELL 8: VIZ 3 — Annual Rainfall Bar ─────────────────────────────────────
rain_city_year = df.groupby(['city','year'])['precipitation_mm'].sum().reset_index()
rain_avg = rain_city_year.groupby('city')['precipitation_mm'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12,6))
bars = ax.bar(rain_avg.index, rain_avg.values,
              color=[city_pal[c] for c in rain_avg.index],
              edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, rain_avg.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
            f'{val:.0f}mm', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Average Annual Rainfall by City (2018–2023)', fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('City'); ax.set_ylabel('Precipitation (mm/year)')
plt.xticks(rotation=30, ha='right')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../visualizations/viz3_annual_rainfall_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💡 Insight: Singapore receives the most rainfall (~{rain_avg['Singapore']:.0f}mm/yr),")
print(f"   while Cairo and Dubai are the driest (<{max(rain_avg['Cairo'],rain_avg['Dubai']):.0f}mm/yr).")


In [ ]:
# ─── CELL 9: VIZ 4 — Seasonal Rainfall ───────────────────────────────────────
season_order = ['Spring','Summer','Autumn','Winter']
s_colors = ['#2ecc71','#e84545','#f39c12','#3498db']

fig, axes = plt.subplots(2, 5, figsize=(16,8))
axes = axes.flatten()
for i, city in enumerate(df['city'].unique()):
    cdf = df[df['city']==city]
    s_rain = cdf.groupby('season')['precipitation_mm'].sum().reindex(season_order) / 6
    axes[i].bar(season_order, s_rain.values, color=s_colors, edgecolor='white')
    axes[i].set_title(city, fontsize=10, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)
    axes[i].set_ylabel('mm/yr', fontsize=8)
    axes[i].grid(axis='y', alpha=0.3)
fig.suptitle('Seasonal Rainfall Distribution by City', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../visualizations/viz4_seasonal_rainfall.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Insight: Mumbai's rainfall peaks sharply in Summer (monsoon season),")
print("   while London has relatively uniform distribution across seasons.")


In [ ]:
# ─── CELL 10: VIZ 8 — Yearly Rainfall Trend ──────────────────────────────────
rain_yr = df.groupby(['year','city'])['precipitation_mm'].sum().reset_index()

fig, ax = plt.subplots(figsize=(13,6))
for city, grp in rain_yr.groupby('city'):
    ax.plot(grp['year'], grp['precipitation_mm'], marker='s',
            linewidth=2, label=city, color=city_pal[city])
ax.set_title('Year-on-Year Total Rainfall Trend by City', fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('Year'); ax.set_ylabel('Total Precipitation (mm)')
ax.legend(ncol=5, fontsize=9, loc='upper center', bbox_to_anchor=(0.5,-0.18))
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('../visualizations/viz8_yearly_rainfall_trend.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Extreme Weather Events

In [ ]:
# ─── CELL 11: VIZ 7 — Extreme Weather Events ─────────────────────────────────
extreme = df[df['weather_condition'].isin(['Heavy Rain','Extreme Heat','Snow/Frost'])]
event_city = extreme.groupby(['city','weather_condition']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12,6))
event_city.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', linewidth=0.8)
ax.set_title('Extreme Weather Events by City (2018–2023)', fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('City'); ax.set_ylabel('Number of Days')
plt.xticks(rotation=30, ha='right')
ax.legend(title='Event Type', fontsize=9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../visualizations/viz7_extreme_weather_events.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nExtreme Event Summary:")
print(event_city.to_string())
print(f"\n💡 Insight: Moscow recorded the most Snow/Frost days ({event_city.get('Snow/Frost', pd.Series()).get('Moscow', 0)}).")
print("   Singapore and Mumbai lead in Heavy Rain events (monsoon/tropical).")


## 6. Weather Variable Correlations & Wind Analysis

In [ ]:
# ─── CELL 12: VIZ 9 — Correlation Heatmap ────────────────────────────────────
cols = ['temp_avg_c','temp_max_c','temp_min_c','precipitation_mm',
        'humidity_pct','wind_speed_kmh','pressure_hpa','uv_index','visibility_km']
corr = df[cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label':'Pearson r'}, vmin=-1, vmax=1)
ax.set_title('Weather Variables Correlation Matrix', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../visualizations/viz9_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key Correlations:")
print(f"  Temp↔UV Index     : {corr.loc['temp_avg_c','uv_index']:.2f}")
print(f"  Humidity↔Rain     : {corr.loc['humidity_pct','precipitation_mm']:.2f}")
print(f"  Pressure↔Temp     : {corr.loc['pressure_hpa','temp_avg_c']:.2f}")
print(f"  Visibility↔Rain   : {corr.loc['visibility_km','precipitation_mm']:.2f}")


In [ ]:
# ─── CELL 13: VIZ 6 — Humidity vs Rainfall ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10,7))
for city in df['city'].unique():
    cdf = df[df['city']==city].sample(200, random_state=42)
    ax.scatter(cdf['humidity_pct'], cdf['precipitation_mm'],
               alpha=0.45, s=25, label=city, color=city_pal[city])
x = df['humidity_pct'].values; y = df['precipitation_mm'].values
slope, intercept, r, p, _ = stats.linregress(x, y)
xl = np.linspace(x.min(), x.max(), 100)
ax.plot(xl, slope*xl+intercept, 'k--', linewidth=1.5, label=f'Trend (r={r:.2f})')
ax.set_title('Humidity vs Daily Precipitation', fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('Humidity (%)'); ax.set_ylabel('Precipitation (mm)')
ax.legend(ncol=4, fontsize=8, loc='upper center', bbox_to_anchor=(0.5,-0.18))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/viz6_humidity_rainfall_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💡 Pearson r (Humidity vs Rainfall) = {r:.2f} — moderate positive relationship")


In [ ]:
# ─── CELL 14: VIZ 10 — Wind Speed Distribution ───────────────────────────────
fig, ax = plt.subplots(figsize=(12,6))
for city in df['city'].unique():
    df[df['city']==city]['wind_speed_kmh'].plot.kde(
        ax=ax, label=city, color=city_pal[city], linewidth=2)
ax.set_title('Wind Speed Distribution by City (KDE)', fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('Wind Speed (km/h)'); ax.set_ylabel('Density')
ax.set_xlim(0, 40)
ax.legend(ncol=5, fontsize=9, loc='upper center', bbox_to_anchor=(0.5,-0.18))
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('../visualizations/viz10_wind_speed_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Statistical Hypothesis Testing

In [ ]:
# ─── CELL 15: Hypothesis Tests ───────────────────────────────────────────────
print("=" * 60)
print("STATISTICAL HYPOTHESIS TESTING")
print("=" * 60)

# Test 1: Is Mumbai's monsoon rainfall significantly different from other months?
mumbai = df[df['city']=='Mumbai'].copy()
monsoon   = mumbai[mumbai['month'].isin([6,7,8,9])]['precipitation_mm']
non_monsoon = mumbai[~mumbai['month'].isin([6,7,8,9])]['precipitation_mm']
t_stat, p_val = stats.ttest_ind(monsoon, non_monsoon)
print(f"\nTest 1: Mumbai Monsoon vs Non-Monsoon Rainfall")
print(f"  Monsoon avg     : {monsoon.mean():.2f} mm/day")
print(f"  Non-monsoon avg : {non_monsoon.mean():.2f} mm/day")
print(f"  t-statistic     : {t_stat:.3f}")
print(f"  p-value         : {p_val:.6f}")
print(f"  Conclusion      : {'Significant ✅' if p_val < 0.05 else 'Not significant ❌'}")

# Test 2: Pearson correlation — Temperature vs UV Index
r, p = stats.pearsonr(df['temp_avg_c'], df['uv_index'])
print(f"\nTest 2: Temperature vs UV Index Correlation")
print(f"  Pearson r  : {r:.3f}")
print(f"  p-value    : {p:.6f}")
print(f"  Conclusion : {'Significant positive correlation ✅' if p < 0.05 else 'No significant correlation'}")

# Test 3: ANOVA — Do cities have significantly different temperatures?
groups = [df[df['city']==c]['temp_avg_c'].values for c in df['city'].unique()]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"\nTest 3: One-way ANOVA — Temperature across Cities")
print(f"  F-statistic : {f_stat:.2f}")
print(f"  p-value     : {p_anova:.6f}")
print(f"  Conclusion  : {'Significant differences exist ✅' if p_anova < 0.05 else 'No significant difference'}")


## 8. Business Insights & Recommendations

---

### 📊 Key Findings

| Insight | Detail |
|---------|--------|
| **Hottest City** | Dubai (avg ~32°C year-round) |
| **Coldest City** | Moscow (avg ~6°C, drops below -15°C in winter) |
| **Wettest City** | Singapore (~1,095mm/year) |
| **Driest City** | Cairo (~73mm/year) |
| **Highest UV Risk** | Dubai & Singapore |
| **Most Extreme Events** | Moscow (Snow/Frost), Singapore (Heavy Rain) |

---

### 💡 Business Recommendations

1. **Urban Planning**: Moscow and New York must invest in winter infrastructure (snow clearance, heating systems) given 100+ frost days annually.
2. **Agriculture**: Mumbai's sharp monsoon peak (June–September) means crop planning must account for 70%+ of annual rainfall in just 4 months.
3. **Tourism**: London and Sydney offer the most temperate conditions for year-round tourism with moderate and predictable weather.
4. **Renewable Energy**: Dubai's consistently high UV index makes it ideal for solar energy deployment — 11+ months of high solar radiation.
5. **Flood Risk**: Singapore and Mumbai require robust drainage systems due to heavy rain events exceeding 10mm/day frequently.
6. **Health Advisory**: Humidity above 80% combined with high temperatures (Singapore, Mumbai) increases heat stress risk for outdoor workers.

---


In [ ]:
# ─── CELL 16: Final Summary Statistics ───────────────────────────────────────
print("=" * 60)
print("PROJECT 3 — GLOBAL WEATHER ANALYSIS: FINAL SUMMARY")
print("=" * 60)
print(f"  Total records analysed : {len(df):,}")
print(f"  Cities covered         : {df['city'].nunique()}")
print(f"  Time period            : {df['year'].min()} – {df['year'].max()}")
print(f"  Visualizations created : 10")
print(f"  Hypothesis tests run   : 3")
print()
print("Top Statistics:")
print(f"  Global avg temp       : {df['temp_avg_c'].mean():.1f}°C")
print(f"  Hottest day recorded  : {df['temp_max_c'].max():.1f}°C ({df.loc[df['temp_max_c'].idxmax(),'city']})")
print(f"  Coldest day recorded  : {df['temp_min_c'].min():.1f}°C ({df.loc[df['temp_min_c'].idxmin(),'city']})")
print(f"  Highest single-day rain: {df['precipitation_mm'].max():.1f}mm ({df.loc[df['precipitation_mm'].idxmax(),'city']})")
print(f"  Avg global humidity   : {df['humidity_pct'].mean():.1f}%")
print()
print("✅ Analysis complete. All outputs saved to visualizations/ folder.")
